# Laboratorio Clase 10.2 — Redes Neuronales Recurrentes (RNN)

**Curso:** Redes Neuronales (EIN097B) — USM  
**Profesor:** Diego Ramírez  
**Duración:** 2 sesiones de 90 min

## Objetivo
Construir, entrenar y comparar redes recurrentes (SimpleRNN, LSTM, GRU) sobre **series de tiempo** y **texto**, y entender por qué los mecanismos de compuertas (LSTM/GRU) son necesarios para capturar dependencias largas.

## Mapa del notebook
1. Setup y verificación del entorno.
2. Parte 1 — RNN desde cero en NumPy (forward pass).
3. Parte 2 — Experimento de vanishing gradient.
4. Parte 3 — Serie de tiempo sintética: ventanas deslizantes.
5. Parte 4 — Entrenar SimpleRNN sobre la serie sintética.
6. Parte 5 — Comparar SimpleRNN vs LSTM vs GRU.
7. Parte 6 — Serie de tiempo real (Jena Climate) con LSTM apilada.
8. Parte 7 — NLP: clasificación de sentimiento (IMDB) con BiLSTM.
9. Parte 8 (bonus) — Generación de texto carácter-a-carácter.
10. Ejercicios y Proyecto Final.

> **Convención:** las celdas marcadas con `# 🧑‍💻 TODO` deben ser completadas por el/la estudiante.

## 0. Setup del entorno

Fijamos semillas para reproducibilidad y verificamos si hay GPU disponible.

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)
print('GPU disponible:', bool(tf.config.list_physical_devices('GPU')))

---
## Parte 1 — RNN desde cero en NumPy

Implementaremos la pasada hacia adelante de una RNN simple:

$$h_t = \tanh(W_{xh}\,x_t + W_{hh}\,h_{t-1} + b_h)$$
$$y_t = W_{hy}\,h_t + b_y$$

**Idea:** procesar la secuencia paso a paso, reutilizando las mismas matrices de pesos en cada timestep.

In [ ]:
def rnn_forward(X, Wxh, Whh, Why, bh, by):
    """Forward pass de una RNN simple.
    X: (T, input_dim) — secuencia de entrada
    Devuelve H: (T, hidden_dim) e Y: (T, output_dim)
    """
    T, _ = X.shape
    hidden_dim = Whh.shape[0]
    H = np.zeros((T, hidden_dim))
    Y = np.zeros((T, Why.shape[0]))
    h_prev = np.zeros(hidden_dim)
    for t in range(T):
        h_prev = np.tanh(Wxh @ X[t] + Whh @ h_prev + bh)
        H[t] = h_prev
        Y[t] = Why @ h_prev + by
    return H, Y

# Mini-ejemplo: secuencia de 5 pasos, 3 features, 4 unidades ocultas, 2 outputs
T, D, H_DIM, O = 5, 3, 4, 2
rng = np.random.default_rng(SEED)
X = rng.normal(size=(T, D))
Wxh = rng.normal(scale=0.3, size=(H_DIM, D))
Whh = rng.normal(scale=0.3, size=(H_DIM, H_DIM))
Why = rng.normal(scale=0.3, size=(O, H_DIM))
bh = np.zeros(H_DIM); by = np.zeros(O)

H, Y = rnn_forward(X, Wxh, Whh, Why, bh, by)
print('Estados ocultos H:'); print(H.round(3))
print('\nSalidas Y:'); print(Y.round(3))

### Verificación contra Keras
Construimos el mismo modelo en Keras inicializando los pesos a mano y verificamos que el output coincide.

In [ ]:
# Keras usa convención (input, units) y (units, units), traspuesta a la nuestra.
keras_rnn = tf.keras.Sequential([
    layers.SimpleRNN(H_DIM, activation='tanh', return_sequences=True,
                     input_shape=(None, D), use_bias=True),
    layers.Dense(O, use_bias=True)
])
# Asignamos los mismos pesos
keras_rnn.layers[0].set_weights([Wxh.T, Whh.T, bh])
keras_rnn.layers[1].set_weights([Why.T, by])

Y_keras = keras_rnn.predict(X[np.newaxis, ...], verbose=0)[0]
print('Coincide con Keras:', np.allclose(Y, Y_keras, atol=1e-5))

---
## Parte 2 — Experimento de Vanishing Gradient

Simulamos la cadena $\prod_{t=1}^{T} W_{hh}\,\mathrm{diag}(\tanh')$ para ver cómo decae el gradiente con la longitud de la secuencia.

In [ ]:
def gradient_norm(seq_length, w_scale):
    """Norma del gradiente acumulado tras propagar hacia atrás `seq_length` pasos."""
    g = np.eye(8)
    W = np.random.default_rng(0).normal(scale=w_scale, size=(8, 8))
    for _ in range(seq_length):
        # tanh' está acotado en (0, 1] → multiplicamos por algo ~0.8 en promedio
        g = (W * 0.8) @ g
    return np.linalg.norm(g)

lengths = np.arange(1, 101, 2)
norms_small = [gradient_norm(L, 0.5) for L in lengths]   # vanishing
norms_large = [gradient_norm(L, 1.5) for L in lengths]   # exploding

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].semilogy(lengths, norms_small); ax[0].set_title('Vanishing (Wₕₕ ~ N(0, 0.5))')
ax[0].set_xlabel('Longitud secuencia'); ax[0].set_ylabel('‖∂L/∂h₁‖ (log)')
ax[1].semilogy(lengths, norms_large); ax[1].set_title('Exploding (Wₕₕ ~ N(0, 1.5))')
ax[1].set_xlabel('Longitud secuencia')
plt.tight_layout(); plt.show()

**Para discutir:** ¿qué pasa con un gradiente del orden de $10^{-15}$ al actualizar los pesos? ¿Y con uno del orden de $10^{+10}$? Anota tus respuestas en una celda de markdown.

---
## Parte 3 — Serie de tiempo sintética

Generamos una serie como suma de senos + ruido. Es controlable y permite separar la **señal** del **ruido**.

In [ ]:
N = 2000
t = np.linspace(0, 100, N)
signal = np.sin(t) + 0.5 * np.sin(2.5 * t) + 0.1 * np.random.randn(N)

plt.figure(figsize=(11, 3))
plt.plot(signal[:400]); plt.title('Primeros 400 puntos de la serie sintética')
plt.xlabel('t'); plt.ylabel('x(t)'); plt.show()

### Ventanas deslizantes
Transformamos la serie en un dataset supervisado: dada una ventana de `seq_length` pasos, predecir el siguiente valor.

In [ ]:
def make_windows(series, seq_length):
    X, y = [], []
    for i in range(len(series) - seq_length):
        X.append(series[i:i+seq_length])
        y.append(series[i+seq_length])
    return np.array(X)[..., np.newaxis], np.array(y)

SEQ_LEN = 40
X, y = make_windows(signal, SEQ_LEN)

# ⚠️ Split CRONOLÓGICO — sin shuffle
split = int(0.8 * len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]
print('X_train:', X_train.shape, '  X_val:', X_val.shape)

---
## Parte 4 — SimpleRNN baseline

In [ ]:
def build_model(kind, units=32, seq_len=SEQ_LEN):
    inp = layers.Input(shape=(seq_len, 1))
    if kind == 'rnn':   x = layers.SimpleRNN(units)(inp)
    elif kind == 'lstm': x = layers.LSTM(units)(inp)
    elif kind == 'gru':  x = layers.GRU(units)(inp)
    out = layers.Dense(1)(x)
    m = models.Model(inp, out)
    m.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return m

model_rnn = build_model('rnn')
model_rnn.summary()

In [ ]:
hist_rnn = model_rnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20, batch_size=32, verbose=0,
    callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
)
print('Val MAE final:', round(hist_rnn.history['val_mae'][-1], 4))

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(hist_rnn.history['loss'], label='train')
plt.plot(hist_rnn.history['val_loss'], label='val')
plt.title('SimpleRNN — curva de pérdida'); plt.legend(); plt.show()

---
## Parte 5 — Comparación SimpleRNN vs LSTM vs GRU

Entrenamos los tres con la **misma arquitectura externa** y comparamos.

In [ ]:
import time
results = {}
for kind in ['rnn', 'lstm', 'gru']:
    tf.random.set_seed(SEED)
    m = build_model(kind)
    t0 = time.time()
    h = m.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=20, batch_size=32, verbose=0,
              callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
    dt = time.time() - t0
    val_mae = min(h.history['val_mae'])
    results[kind] = {'val_mae': val_mae, 'time_s': dt, 'history': h.history, 'model': m}
    print(f'{kind.upper():6s}  val_MAE={val_mae:.4f}  tiempo={dt:.1f}s')

pd.DataFrame({k: {'val_MAE': v['val_mae'], 'tiempo(s)': v['time_s']}
              for k, v in results.items()}).T

In [ ]:
plt.figure(figsize=(10, 4))
for kind, c in zip(['rnn', 'lstm', 'gru'], ['C0', 'C1', 'C2']):
    plt.plot(results[kind]['history']['val_loss'], c=c, label=f'{kind.upper()}')
plt.title('val_loss por época'); plt.xlabel('época'); plt.legend(); plt.show()

### Predicción visual
Graficamos las predicciones del mejor modelo sobre el set de validación.

In [ ]:
best = min(results, key=lambda k: results[k]['val_mae'])
y_pred = results[best]['model'].predict(X_val, verbose=0).ravel()

plt.figure(figsize=(12, 3))
plt.plot(y_val[:300], label='real')
plt.plot(y_pred[:300], label=f'predicción ({best.upper()})', alpha=0.8)
plt.legend(); plt.title(f'Mejor modelo: {best.upper()}'); plt.show()

---
## Parte 6 — Serie de tiempo real: Jena Climate

Dataset oficial del tutorial de Keras: lecturas meteorológicas en Jena (2009–2016), tomadas cada 10 min.

In [ ]:
url = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip'
path = tf.keras.utils.get_file('jena_climate_2009_2016.csv.zip', url, extract=True)
csv_path = path.replace('.zip', '')
df = pd.read_csv(csv_path)
df.head()

In [ ]:
# Submuestreamos a 1 lectura por hora (cada 6 filas)
temp = df['T (degC)'].values[5::6]
print('Lecturas horarias:', len(temp))

plt.figure(figsize=(12, 3))
plt.plot(temp[:24*30]); plt.title('Temperatura — primer mes')
plt.xlabel('hora'); plt.ylabel('°C'); plt.show()

In [ ]:
# Normalizamos con estadísticas calculadas SOLO sobre el train
n = len(temp)
train_end = int(0.7 * n); val_end = int(0.85 * n)
mean, std = temp[:train_end].mean(), temp[:train_end].std()
temp_n = (temp - mean) / std

SEQ = 24 * 5  # 5 días de contexto
X, y = make_windows(temp_n, SEQ)
X_tr, y_tr = X[:train_end-SEQ], y[:train_end-SEQ]
X_va, y_va = X[train_end-SEQ:val_end-SEQ], y[train_end-SEQ:val_end-SEQ]
X_te, y_te = X[val_end-SEQ:], y[val_end-SEQ:]
print('train:', X_tr.shape, ' val:', X_va.shape, ' test:', X_te.shape)

In [ ]:
model_jena = models.Sequential([
    layers.Input(shape=(SEQ, 1)),
    layers.LSTM(32, return_sequences=True),
    layers.LSTM(32),
    layers.Dense(1)
])
model_jena.compile(optimizer='adam', loss='mse', metrics=['mae'])
model_jena.summary()

In [ ]:
hist_jena = model_jena.fit(
    X_tr, y_tr, validation_data=(X_va, y_va),
    epochs=10, batch_size=128, verbose=1,
    callbacks=[
        callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(patience=2, factor=0.5),
    ]
)

In [ ]:
# Baseline ingenuo: predecir y_t = y_{t-1}
naive_mae = np.mean(np.abs(y_te - X_te[:, -1, 0])) * std
model_mae = model_jena.evaluate(X_te, y_te, verbose=0)[1] * std
print(f'Baseline ingenuo  MAE = {naive_mae:.3f} °C')
print(f'LSTM (2 capas)    MAE = {model_mae:.3f} °C')

**Discusión:** ¿el modelo le gana al baseline ingenuo? ¿En qué horizontes? Comenta en una celda de markdown.

---
## Parte 7 — NLP: Clasificación de sentimiento (IMDB)

Dataset clásico: reseñas de películas etiquetadas como positivas o negativas.

In [ ]:
VOCAB = 10_000
MAXLEN = 200

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(num_words=VOCAB)
x_train = tf.keras.utils.pad_sequences(x_train, maxlen=MAXLEN)
x_test  = tf.keras.utils.pad_sequences(x_test,  maxlen=MAXLEN)
print('train:', x_train.shape, ' test:', x_test.shape)

In [ ]:
model_imdb = models.Sequential([
    layers.Embedding(VOCAB, 64),
    layers.Bidirectional(layers.LSTM(32, dropout=0.3, recurrent_dropout=0.0)),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid'),
])
model_imdb.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_imdb.summary()

In [ ]:
hist_imdb = model_imdb.fit(
    x_train, y_train, validation_split=0.1,
    epochs=4, batch_size=128, verbose=1,
    callbacks=[callbacks.EarlyStopping(patience=2, restore_best_weights=True)],
)
loss, acc = model_imdb.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy: {acc:.3f}')

---
## Parte 8 (bonus) — Generación de texto carácter-a-carácter

Entrenamos una LSTM para predecir el siguiente carácter de un texto. Sirve para ver cómo una RNN **modela una distribución** y permite muestrear de ella.

In [ ]:
url = 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
path = tf.keras.utils.get_file('shakespeare.txt', url)
text = open(path, 'r').read()[:200_000]   # recortamos para que entrene rápido
chars = sorted(set(text))
c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for c, i in c2i.items()}
print(f'{len(text):,} caracteres, {len(chars)} únicos')

In [ ]:
SEQ = 60
STEP = 3   # tomamos una ventana cada 3 caracteres para no explotar memoria
X = np.zeros((len(text) // STEP, SEQ), dtype=np.int32)
y = np.zeros(len(text) // STEP, dtype=np.int32)
k = 0
for i in range(0, len(text) - SEQ - 1, STEP):
    X[k] = [c2i[c] for c in text[i:i+SEQ]]
    y[k] = c2i[text[i+SEQ]]
    k += 1
X, y = X[:k], y[:k]
print('Ejemplos:', X.shape)

In [ ]:
model_gen = models.Sequential([
    layers.Embedding(len(chars), 32, input_length=SEQ),
    layers.LSTM(128),
    layers.Dense(len(chars), activation='softmax'),
])
model_gen.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model_gen.fit(X, y, batch_size=128, epochs=5, verbose=1)

In [ ]:
def sample(seed, n=400, temperature=0.8):
    out = seed
    for _ in range(n):
        x = np.array([[c2i.get(c, 0) for c in out[-SEQ:].rjust(SEQ)]])
        probs = model_gen.predict(x, verbose=0)[0]
        probs = np.log(probs + 1e-9) / temperature
        probs = np.exp(probs) / np.exp(probs).sum()
        out += i2c[np.random.choice(len(chars), p=probs)]
    return out

print(sample('ROMEO: ', n=300, temperature=0.7))

**Para experimentar:** prueba `temperature = 0.2`, `0.8`, `1.5`. ¿Qué cambia? ¿Por qué?

---
# 📝 Ejercicios

## Ejercicio 1 — Forward pass a mano (10 pts)
Modifica la función `rnn_forward` para que también devuelva el promedio de `|h_t|` en cada paso (norma L1 promedio de los estados ocultos). Aplícala a una secuencia de 100 pasos con `Wxh, Whh` inicializados con escala `0.5` y luego con escala `1.5`. Grafica ambos resultados y discute qué observas (¿se saturan? ¿divergen?).

## Ejercicio 2 — Comparación SimpleRNN / LSTM / GRU (15 pts)
Sobre la serie sintética de la Parte 3, **alarga la dependencia**: cambia la serie a `sin(t) + sin(t/20)` para crear estacionalidades muy largas. Reentrena los 3 modelos con `seq_length = 100`. Reporta MAE de validación y tiempo de entrenamiento en una tabla. ¿Cuál modelo se beneficia más al alargar la dependencia? ¿Por qué?

## Ejercicio 3 — Ventana de contexto (10 pts)
Sobre Jena Climate, prueba `SEQ ∈ {24, 24*3, 24*7}` (1 día, 3 días, 7 días). Reporta MAE en grados Celsius para cada uno. ¿Existe un punto de saturación? Justifica.

## Ejercicio 4 — BiLSTM + Dropout en IMDB (10 pts)
Entrena 3 variantes del modelo IMDB y reporta accuracy en test:
1. `LSTM(32)` (unidireccional, sin dropout).
2. `Bidirectional(LSTM(32))` (sin dropout).
3. `Bidirectional(LSTM(32, dropout=0.3))` + `Dropout(0.4)` antes de la salida.

Comenta el efecto de cada cambio sobre `train_acc` vs `val_acc`.

---

# 🎓 Proyecto Final

Elige un dataset de la tabla de la guía (`Guia_Laboratorio_RNN.md`, sección 6) o propón uno propio (avisar al docente). Entrega un análisis completo con:

1. **Introducción** y motivación de la tarea elegida.
2. **Exploración** (estacionalidad / autocorrelación para series; distribución de largos para texto).
3. **Preprocesamiento** con split temporal sin fugas.
4. **Modelo RNN propio**, justificando: tipo de celda, # de capas, # de unidades, regularización.
5. **Entrenamiento** con `EarlyStopping` y `ReduceLROnPlateau`, curvas anotadas.
6. **Evaluación** con la métrica adecuada y comparación contra un **baseline** (ingenuo para series, regresión logística + TF-IDF para texto).
7. **Conclusiones**: limitaciones de tu modelo, ideas que probarías a continuación.

**Puntos extra:**
- (+5) Implementa una capa de **atención** simple sobre los estados ocultos y compara.
- (+5) Compara contra un baseline clásico (ARIMA / Holt-Winters o TF-IDF + LogReg).

---

*Fin del laboratorio — Clase 10.2 RNN — USM, junio 2026.*